# ⚡ Notebook 10: Complete End-to-End Inference Pipeline
## Standalone Execution Pipeline: Preprocessing, Transformation, Classification & Anomaly Check

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

from src.utils import load_model, load_numpy
from src.evaluation_metrics import compute_all_metrics

X_test = load_numpy(os.path.join(PROJECT_ROOT, "data", "processed", "X_test.npy"))
y_test_bin = load_numpy(os.path.join(PROJECT_ROOT, "data", "processed", "y_test_binary.npy"))
y_test_multi = load_numpy(os.path.join(PROJECT_ROOT, "data", "processed", "y_test_multi.npy"))
y_test_sev = load_numpy(os.path.join(PROJECT_ROOT, "data", "processed", "y_test_severity.npy"))

preprocessor = load_model(os.path.join(PROJECT_ROOT, "models", "encoders", "preprocessor.pkl"))
le_multi = load_model(os.path.join(PROJECT_ROOT, "models", "encoders", "label_encoder_multi.pkl"))

best_clf = load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "xgboost_model.pkl"))
multi_clf = load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "svm_multiclass_ovr.pkl"))
sev_reg = load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "svr_model.pkl"))
dbscan = load_model(os.path.join(PROJECT_ROOT, "models", "saved_models", "dbscan_model.pkl"))

In [ ]:
sample_indices = [10, 50, 100, 250, 500, 1000, 1500, 2000]
X_samples = X_test[sample_indices]
y_true_binary = y_test_bin[sample_indices]
y_true_categories = [le_multi.classes_[i] for i in y_test_multi[sample_indices]]
y_true_severities = y_test_sev[sample_indices]

pred_bin = best_clf.predict(X_samples)
pred_multi_idx = multi_clf.predict(X_samples)
pred_categories = [le_multi.classes_[i] for i in pred_multi_idx]
pred_severities = sev_reg.predict(X_samples)

pipeline_demo_df = pd.DataFrame({
    "Sample Index": sample_indices,
    "True Label": ["Attack" if y == 1 else "Normal" for y in y_true_binary],
    "Pred Label": ["Attack" if y == 1 else "Normal" for y in pred_bin],
    "True Category": y_true_categories,
    "Pred Category": pred_categories,
    "True Severity": y_true_severities,
    "Pred Severity": np.round(pred_severities, 3)
})

print(pipeline_demo_df.to_string(index=False))

In [ ]:
full_preds = best_clf.predict(X_test)
full_probs = best_clf.predict_proba(X_test)[:, 1] if hasattr(best_clf, "predict_proba") else None
final_metrics = compute_all_metrics(y_test_bin, full_preds, full_probs)

print("Pipeline Production Performance Summary:")
for k, v in final_metrics.items():
    if k != "confusion_matrix":
        print(f"  {k:15s}: {v:.4f}")